# bge-m3
Embed the legal chunks with one model and score it on `eval_questions.json`.
To test another model, copy this notebook and change only the config cell.

In [ ]:
import sys
sys.path.insert(0, "..")  # common.py lives one folder up
import pandas as pd
import common

# ---- config (the only cell that differs between model notebooks) ----
NAME = "bge-m3"                 # this folder's name (cache goes here) and row name in results.csv
MODEL_NAME = "BAAI/bge-m3"
QUERY_PREFIX = ""               # e5 models need "query: "
PASSAGE_PREFIX = ""             # e5 models need "passage: "
MAX_LENGTH = 1024
BATCH_SIZE = 16
FORCE = False                   # True = ignore the cache and re-embed

## 1. Chunks

In [ ]:
articles = common.load_articles()
chunks = common.build_chunks(articles)
questions = common.load_questions()
print(f"{len(articles)} articles -> {len(chunks)} chunks, {len(questions)} questions")
print("chunking:", common.CHUNKING_VERSION)
print("longest chunk (chars):", max(len(c["embed_text"]) for c in chunks))
print(chunks[0]["embed_text"])

## 2. Embed + evaluate
Vectors are cached in this folder (`vectors.npy` + `chunks.json` + `info.json`), so re-running is fast.

In [ ]:
result = common.run_experiment(
    NAME, MODEL_NAME, chunks, questions,
    query_prefix=QUERY_PREFIX, passage_prefix=PASSAGE_PREFIX,
    max_length=MAX_LENGTH, batch_size=BATCH_SIZE, force=FORCE,
)
print(result["info"])
result["metrics"]

## 3. Misses
Questions where no relevant article is in the top 5 (or ranked low). Check whether the question or the label is wrong before blaming the model.

In [ ]:
df = pd.DataFrame(result["details"])
df[df["rank"].isna() | (df["rank"] > 1)][["id", "question", "relevant", "rank", "top5"]]

## 4. All models so far

In [ ]:
pd.read_csv(common.RESULTS_PATH).sort_values("mrr", ascending=False)